# SuperDubDub · Phase 0: Benchmark

Đo tốc độ và chất lượng từng bước trên GPU T4 miễn phí, trước khi viết pipeline đầy đủ.

**Chuẩn bị (làm 1 lần):**
1. `Runtime → Change runtime type → T4 GPU`.
2. Biểu tượng 🔑 bên trái → thêm secret `GEMINI_API_KEY`, bật **Notebook access**.
3. Video nằm trong `Google Drive/SuperDubDub/input/`.

**Chạy lần lượt ô 1 → 5.** Ô nào báo lỗi: copy toàn bộ phần log đỏ gửi cho Claude.
Cuối cùng ô 5 in ra một đoạn JSON: copy gửi cho Claude để chốt engine.

In [ ]:
#@title 1. Cài đặt (mỗi phiên Colab, ~5–10 phút)
BRANCH = "feat/phase0-notebook"  #@param {type:"string"}
GEMINI_MODEL = "gemini-2.5-flash"  #@param {type:"string"}
import os, shutil, subprocess, sys, time
from google.colab import drive, userdata

if not shutil.which("nvidia-smi") or subprocess.run(["nvidia-smi"], capture_output=True).returncode != 0:
    raise SystemExit("Chưa có GPU: Runtime → Change runtime type → T4 GPU, rồi chạy lại ô này.")
t0 = time.time()
drive.mount("/content/drive")
REPO = "/content/MinhKhoiDubDub"
if os.path.isdir(REPO):
    !git -C {REPO} fetch -q origin {BRANCH} && git -C {REPO} checkout -q -B {BRANCH} FETCH_HEAD
else:
    !git clone -q -b {BRANCH} https://github.com/daominhkhoi/MinhKhoiDubDub.git {REPO}
!git -C {REPO} log --oneline -1
sys.path.insert(0, REPO)
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
!bash {REPO}/colab/setup.sh asr tts
os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")
os.environ["GEMINI_MODEL"] = GEMINI_MODEL
print(f"Cài đặt xong sau {time.time() - t0:.0f} giây")

In [ ]:
#@title 2. Nhận dạng giọng nói (WhisperX)
VIDEO = "SQL Server 2022 - Set Up A -Dummy- Database.mp4"  #@param {type:"string"}
SRC_LANG = "en"  #@param ["en", "ja", "zh"]
MINUTES = 5  #@param {type:"number"}
ASR_MODEL = "large-v3"  #@param ["large-v3", "large-v3-turbo"]
from dub import bench
bench.asr(VIDEO, SRC_LANG, MINUTES, ASR_MODEL)

In [ ]:
#@title 3. Dịch bằng Gemini (2 pass)
N_SEGMENTS = 50  #@param {type:"integer"}
bench.translate(VIDEO, N_SEGMENTS)

In [ ]:
#@title 4. Nghe mù 3 giọng đọc (viXTTS + 2 giọng edge-tts)
N_SENTENCES = 10  #@param {type:"integer"}
VOICE_REF = ""  #@param {type:"string"}
#@markdown Để trống `VOICE_REF` thì viXTTS clone giọng người nói trong video. Hoặc nhập đường dẫn file wav 6–12 giây, ví dụ `/content/drive/MyDrive/SuperDubDub/voices/nam.wav`.
bench.tts(VIDEO, N_SENTENCES, VOICE_REF)

In [ ]:
#@title 5. Chấm điểm (trước khi xem đáp án), rồi lưu kết quả
SCORE_X = 3  #@param {type:"slider", min:1, max:5, step:1}
SCORE_Y = 3  #@param {type:"slider", min:1, max:5, step:1}
SCORE_Z = 3  #@param {type:"slider", min:1, max:5, step:1}
bench.RESULTS["tts"]["scores"] = {"Giọng X": SCORE_X, "Giọng Y": SCORE_Y, "Giọng Z": SCORE_Z}
bench.reveal()
bench.save_report()